# mboostrs — performance vs. R

Same model, same data, two implementations: R's mboost (interpreted R loop over R closures, internal C for least-squares) and the Rust core (released GIL, BLAS-3 weighted Gram, scalar kernel fast path).

The embedded cross-language tables are a **legacy 2026-06-11 snapshot**, kept
only to illustrate visualization. They predate the structured benchmark contract
and must not be used as current performance evidence.

Workloads:
1. glmboost Gaussian on (n, p) grids of varying size.
2. gamboost with two bbs(...) P-spline base-learners.
3. gamboost non-Gaussian (Binomial, Poisson, Laplace).
4. cvrisk (B=5 folds) on a moderate-size model.
5. Live microbenchmark — fit a model in the notebook.
6. Scaling with mstop.

Current evidence is generated by `python benches/bench_all.py --output benchmark.json`.
That runner records raw repetitions and provenance, enforces equal boundaries and
matching thread modes, and rejects ratios whose fitted outputs fail correctness.

In [1]:
import time
import statistics
import numpy as np
import polars as pl
import altair as alt
import pandas as pd

import mboostrs as mb
print('mboostrs:', mb.__version__)

## 1. glmboost Gaussian, varying n and p

mstop=200, nu=0.1, centered design (R default). The Rust engine fuses the per-iteration (\u03b2\u2c7c, SSE\u2c7c) calculation across all predictors into one tight loop — per-step cost is roughly O(n\u00b7p) with a small constant.

In [10]:
# Full benchmark code in benches/bench_all.py
# Results from 2026-06-11 (Apple M-series, mstop=200, nu=0.1):
# fmt: off
glm_data = pl.DataFrame({
    "config": ["n=500\np=5", "n=500\np=25", "n=2000\np=10", "n=2000\np=50",
               "n=5000\np=20", "n=10000\np=20", "n=10000\np=50", "n=20000\np=20"],
    "r_med_ms": [7.9, 12.7, 23.9, 53.3, 64.3, 166.1, 222.2, 283.2],
    "rust_med_ms": [0.4, 1.2, 2.0, 6.5, 7.4, 14.9, 32.7, 30.1],
    "speedup": [20.6, 11.0, 12.1, 8.2, 8.7, 11.1, 6.8, 9.4],
})
# fmt: on
display(glm_data)

pdf = glm_data.to_pandas().melt(
    id_vars=["config", "speedup"],
    value_vars=["r_med_ms", "rust_med_ms"],
    var_name="implementation",
    value_name="time_ms",
)
pdf["implementation"] = pdf["implementation"].map({
    "r_med_ms": "R mboost", "rust_med_ms": "mboostrs (Rust)"
})

# Position speedup label at the taller bar in each group
pdf["is_higher"] = pdf.groupby("config")["time_ms"].transform(lambda x: x == x.max())
speedup_map = dict(zip(glm_data["config"].to_list(), glm_data["speedup"].map_elements(lambda x: f"{x}x").to_list()))
pdf["speedup_str"] = ""
pdf.loc[pdf["is_higher"], "speedup_str"] = pdf.loc[pdf["is_higher"], "config"].map(speedup_map)

bars = alt.Chart(pdf).mark_bar().encode(
    x=alt.X("config:N", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("time_ms:Q", title="median fit time (ms)", scale=alt.Scale(type="log")),
    color=alt.Color("implementation:N", scale=alt.Scale(
        domain=["R mboost", "mboostrs (Rust)"],
        range=["#1f77b4", "#d62728"],
    )),
    xOffset="implementation:N",
).properties(title="glmboost Gaussian, mstop=200 \u2014 wall-clock", width=500, height=220)

speedup_labels = alt.Chart(pdf).mark_text(dy=-10, fontSize=10, fontWeight="bold").encode(
    x=alt.X("config:N", title=None),
    y=alt.Y("time_ms:Q", scale=alt.Scale(type="log")),
    xOffset="implementation:N",
    text="speedup_str:N",
)

bars + speedup_labels


## 2. gamboost with bbs(x, df=4) + bbs(z, df=4)

The spline path is dominated by the per-iteration penalised normal equation. Our bbs builds the centered eigenbasis transform once and caches the Cholesky of X\u1d40WX + \u03bbK, so each step is one dense triangular solve.

In [3]:
# Full benchmark code in benches/bench_all.py
# Results from 2026-06-11 (Apple M-series, mstop=200, nu=0.1):
# fmt: off
gam_data = pl.DataFrame({
    "config": ["n=500", "n=2000", "n=5000", "n=10000"],
    "r_med_ms": [36.0, 45.1, 80.3, 136.0],
    "rust_med_ms": [1.4, 2.7, 5.8, 10.1],
    "speedup": [25.2, 16.5, 13.9, 13.4],
})
# fmt: on
display(gam_data)

pdf = gam_data.to_pandas().melt(
    id_vars=["config", "speedup"],
    value_vars=["r_med_ms", "rust_med_ms"],
    var_name="implementation",
    value_name="time_ms",
)
pdf["implementation"] = pdf["implementation"].map({
    "r_med_ms": "R mboost", "rust_med_ms": "mboostrs (Rust)"
})

# Position speedup label at the taller bar in each group
pdf["is_higher"] = pdf.groupby("config")["time_ms"].transform(lambda x: x == x.max())
speedup_map = dict(zip(gam_data["config"].to_list(), gam_data["speedup"].map_elements(lambda x: f"{x}x").to_list()))
pdf["speedup_str"] = ""
pdf.loc[pdf["is_higher"], "speedup_str"] = pdf.loc[pdf["is_higher"], "config"].map(speedup_map)

bars = alt.Chart(pdf).mark_bar().encode(
    x=alt.X("config:N", title=None),
    y=alt.Y("time_ms:Q", title="median fit time (ms)", scale=alt.Scale(type="log")),
    color=alt.Color("implementation:N", scale=alt.Scale(
        domain=["R mboost", "mboostrs (Rust)"],
        range=["#1f77b4", "#d62728"],
    )),
    xOffset="implementation:N",
).properties(title="gamboost y ~ bbs(x) + bbs(z), mstop=200", width=400, height=220)

speedup_labels = alt.Chart(pdf).mark_text(dy=-10, fontSize=10, fontWeight="bold").encode(
    x=alt.X("config:N"),
    y=alt.Y("time_ms:Q", scale=alt.Scale(type="log")),
    xOffset="implementation:N",
    text="speedup_str:N",
)

bars + speedup_labels


## 3. gamboost non-Gaussian families

Binomial, Poisson, and Laplace with bbs(x, df=4) + bbs(z, df=4). These fall back to the general per-iteration normal-equation path (no Gram shortcut), so speedups are smaller.

In [4]:
# Full benchmark code in benches/bench_all.py
# Results from 2026-06-11 (Apple M-series, mstop=200, nu=0.1):
# fmt: off
ng_data = pl.DataFrame({
    "family": ["Binomial", "Binomial", "Poisson", "Poisson", "Laplace", "Laplace"],
    "n": [2000, 5000, 2000, 5000, 2000, 5000],
    "r_med_s": [0.0657, 0.1894, 0.0836, 0.1152, 0.0362, 0.0697],
    "rust_med_s": [0.0170, 0.0418, 0.0164, 0.0355, 0.0114, 0.0272],
    "speedup": [3.9, 4.5, 5.1, 3.2, 3.2, 2.6],
})
# fmt: on
display(ng_data)

pdf = ng_data.to_pandas().melt(
    id_vars=["family", "n", "speedup"],
    value_vars=["r_med_s", "rust_med_s"],
    var_name="implementation",
    value_name="time_s",
)
pdf["implementation"] = pdf["implementation"].map({
    "r_med_s": "R mboost", "rust_med_s": "mboostrs (Rust)"
})
pdf["config"] = pdf["family"] + " n=" + pdf["n"].astype(str)
bars = alt.Chart(pdf).mark_bar().encode(
    x=alt.X("config:N", title=None, sort=list(pdf["config"].unique())),
    y=alt.Y("time_s:Q", title="median fit time (s)"),
    color=alt.Color("implementation:N", scale=alt.Scale(
        domain=["R mboost", "mboostrs (Rust)"],
        range=["#1f77b4", "#d62728"],
    )),
    xOffset="implementation:N",
).properties(title="gamboost non-Gaussian, mstop=200", width=500, height=220)

bars

## 4. Cross-validated risk

cvrisk with B=5 k-fold folds, glmboost Gaussian. R uses papply=lapply (serial); Rust parallelizes across folds with Rayon.

In [5]:
# Full benchmark code in benches/bench_all.py
# Results from 2026-06-11 (Apple M-series, mstop=200, nu=0.1, B=5):
# fmt: off
cv_data = pl.DataFrame({
    "n": [1000, 3000, 5000],
    "p": [10, 20, 20],
    "B": [5, 5, 5],
    "r_med_s": [0.0513, 0.2266, 0.3519],
    "rust_med_s": [0.0027, 0.0114, 0.0183],
    "speedup": [19.3, 19.8, 19.2],
})
# fmt: on
display(cv_data)

pdf = cv_data.to_pandas().melt(
    id_vars=["n", "p", "B", "speedup"],
    value_vars=["r_med_s", "rust_med_s"],
    var_name="implementation",
    value_name="time_s",
)
pdf["implementation"] = pdf["implementation"].map({
    "r_med_s": "R mboost", "rust_med_s": "mboostrs (Rust)"
})
pdf["config"] = "n=" + pdf["n"].astype(str) + " p=" + pdf["p"].astype(str)
bars = alt.Chart(pdf).mark_bar().encode(
    x=alt.X("config:N", title=None, sort=list(pdf["config"].unique())),
    y=alt.Y("time_s:Q", title="median fit time (s)"),
    color=alt.Color("implementation:N", scale=alt.Scale(
        domain=["R mboost", "mboostrs (Rust)"],
        range=["#1f77b4", "#d62728"],
    )),
    xOffset="implementation:N",
).properties(title="cvrisk glmboost Gaussian, B=5", width=400, height=220)

bars


## 5. Live microbenchmark

A glmboost Gaussian fit at six sizes to show scaling characteristics live. No R needed.

In [6]:
rng = np.random.default_rng(42)

def best_of(fn, n_reps=5, warmup=2):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return statistics.median(times), min(times)

def time_glm(n, p, mstop=200, nu=0.1, n_reps=5):
    X = rng.standard_normal((n, p))
    y = rng.standard_normal(n)
    cols = [f"x{i}" for i in range(p)]
    df = pd.DataFrame(X, columns=cols)
    df["y"] = y
    formula = "y ~ " + " + ".join(f"bols({c}, centered=TRUE)" for c in cols)
    def fit():
        model = mb.glmboost(formula, data=df, family=mb.Gaussian(),
                            control=mb.boost_control(mstop=mstop, nu=nu))
    return best_of(fit, n_reps=n_reps)

sizes = [(500, 5), (500, 25), (2000, 10), (2000, 50),
         (5000, 20), (10000, 20)]
rows = []
for n, p in sizes:
    med, bes = time_glm(n, p)
    rows.append((n, p, round(med * 1000, 2), round(bes * 1000, 2)))

live_data = pl.DataFrame(
    rows, schema=["n", "p", "med_ms", "best_ms"]
).with_columns(
    (pl.col("n") * pl.col("p") * 1e-6).alias("n_p_millions")
)
display(live_data)


In [ ]:
# Live scaling chart
ldf = live_data.to_pandas()
alt.Chart(ldf).mark_line(point=True).encode(
    x=alt.X("n_p_millions:Q", title="n\u00d7p (millions)"),
    y=alt.Y("med_ms:Q", title="median fit time (ms)"),
).properties(title="Live microbenchmark: glmboost scaling with n\u00d7p",
    width=400, height=220)


The live timings show sub-linear scaling in n\u00b7p — the fused bulk kernel keeps per-iteration overhead tiny as the problem grows.

**Caveat:** the first call to glmboost includes Rust compilation + import overhead for the PyO3 extension. The warmup rounds absorb that.

## 6. Scaling with mstop

The Rust engine's O(mstop \u00b7 n \u00b7 p) scaling is linear: doubling mstop doubles fit time. R's overhead per iteration (dispatch, closure calls) adds a larger constant.

In [7]:
# Results from benches/bench_all.py
# n=2000, p=10, nu=0.1, glmboost Gaussian
# fmt: off
mstop_data = pl.DataFrame({
    "mstop": [100, 200, 500, 1000, 2000],
    "r_med_ms": [12.1, 23.9, 59.2, 118.8, 237.5],
    "rust_med_ms": [1.0, 2.0, 5.0, 10.1, 20.3],
})
# fmt: on
mstop_data = mstop_data.with_columns(
    (mstop_data["r_med_ms"] / mstop_data["rust_med_ms"]).alias("speedup")
)
display(mstop_data)

pdf = mstop_data.to_pandas().melt(
    id_vars=["mstop", "speedup"],
    value_vars=["r_med_ms", "rust_med_ms"],
    var_name="implementation",
    value_name="time_ms",
)
pdf["implementation"] = pdf["implementation"].map({
    "r_med_ms": "R mboost", "rust_med_ms": "mboostrs (Rust)"
})
lines = alt.Chart(pdf).mark_line(point=True).encode(
    x=alt.X("mstop:Q", title="boosting iterations"),
    y=alt.Y("time_ms:Q", title="median fit time (ms)"),
    color=alt.Color("implementation:N", scale=alt.Scale(
        domain=["R mboost", "mboostrs (Rust)"],
        range=["#1f77b4", "#d62728"],
    )),
).properties(title="glmboost scaling with mstop (n=2000, p=10)", width=400, height=220)
lines


Rust achieves near-constant ~10\u00b5s per iteration at this problem size. R's per-iteration cost is ~120\u00b5s — an order of magnitude higher, dominated by dispatch overhead.

## Summary

All workloads combined:

In [8]:
import numpy as np

all_speedups = (
    list(glm_data["speedup"]) + list(gam_data["speedup"]) 
    + list(ng_data["speedup"]) + list(cv_data["speedup"])
)
geo = np.exp(np.mean(np.log(all_speedups)))
print(f"Legacy, non-publishable geometric mean: {geo:.1f}x")
print(f"Speedup range: {min(all_speedups):.1f}x \u2013 {max(all_speedups):.1f}x")
print(f"Number of benchmark configs: {len(all_speedups)}")

### Caveats

- Wall-clock medians on a single Apple M-series machine; ratios vary with hardware (memory bandwidth dominates the scalar kernel) and BLAS threading.
- The legacy CV row is methodologically invalid: R was serial and already fitted; Python used default parallelism and included the initial fit.
- rpy2 overhead excluded; wall-clock reflects just the fit.
- At very small n\u00b7p, frontend data marshalling and harness overhead can dominate; inspect raw repetitions instead of promoting a ratio.
- Non-Gaussian families use the general normal-equation path (no Gram shortcut); speedups are smaller.
- Live microbenchmarks (section 5) run in Python with no R dependency — verify scaling on your hardware.
- Run `python3 benches/bench_all.py --quick --output benchmark.json` for the checked replacement contract.